## Aggregations a gold

### Objetivo:
    - Obtener informacion sobre "total de presupuesto" y "total de ingresos" de "peliculas", de las peliculas
    - A partir de fecha de lanzamiento 2015 
    - agrupar por "año d el afecha de lanzamiento" y "pais" 
    - ranking ordenado de manera descendente por el "total del presupuesto" y "total de ingresos" particionado por el año de lanzamiento


tables: movie, country, production_country \
columns: year_release_date, budget, revenue, country_name,  \
add: rank 

df -> results_group_movie_country

### Leer archivos

In [0]:
%run "../includes/commom_functions"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date =dbutils.widgets.get("p_file_date")

In [0]:
# %run "../includes/configuration" 

In [0]:
# movies_df = spark.read.parquet(f"{silver_folder_path}/movies")
movies_df = spark.read.table("movie_silver.movies") \
                    .filter(f"file_date = '{v_file_date}'")

In [0]:
# countries_df = spark.read.parquet(f"{silver_folder_path}/countries")
countries_df = spark.read.table("movie_silver.countries")

In [0]:
# productions_countries_df = spark.read.parquet(f"{silver_folder_path}/productions_countries")
productions_countries_df = spark.read.table("movie_silver.productions_countries") \
                    .filter(f"file_date = '{v_file_date}'")

### Join "country" y "productions_country"

In [0]:
countries_prod_countries_df = countries_df.join(productions_countries_df,
                                                    countries_df.country_id == productions_countries_df.country_id,
                                                    "inner") \
                                                .select(countries_df.country_name, productions_countries_df.movie_id)

### Join "production_contries" y "movies_df"

#### - filtrar las peliculas por fecha de lanzamiento mayor o igual a 2015

In [0]:
movies_filtered_df = movies_df.filter("year_release_date >=  2015")

In [0]:
results_movies_countries_df = movies_filtered_df.join(countries_prod_countries_df,
                                                    movies_filtered_df.movie_id == countries_prod_countries_df.movie_id,
                                                    "inner")

In [0]:
results_df = results_movies_countries_df \
    .select("year_release_date", "budget", "revenue", "country_name")

In [0]:
from pyspark.sql.functions import sum, desc, dense_rank, asc, lit
from pyspark.sql.window import Window

In [0]:
results_order_by_df = results_df \
    .groupBy("year_release_date", "country_name") \
    .agg(
        sum("budget").alias("total_budget"),
        sum("revenue").alias("total_revenue")
    )

In [0]:
display(results_order_by_df)

#### - Agregar el ranking

In [0]:
results_dense_rank_df = Window.partitionBy("year_release_date").orderBy(
                                                                desc("total_budget"), 
                                                                desc("total_revenue"))
final_df = results_order_by_df.withColumn("dense_rank", dense_rank().over(results_dense_rank_df)) \
                                .withColumn("created_date", lit(v_file_date))

### Escribir los datos en el datalake en formato "Parquet"

In [0]:
# overwrite_partition("movie_gold", "results_group_movie_country","created_date",v_file_date)

In [0]:
# final_df.write.mode("overwrite").parquet(f"{gold_folder_path}/results_group_movie_country")
# final_df.write.mode("overwrite").format("delta").saveAsTable("movie_gold.results_group_movie_country")

# final_df.write.mode("append").partitionBy("created_date").format("delta").saveAsTable("movie_gold.results_group_movie_country")

In [0]:
# hago una validacion con merge

merge_condition = 'tgt.year_release_date = src.year_release_date AND tgt.country_name = src.country_name AND tgt.created_date=src.created_date'
merge_delta_lake(final_df, "movie_gold", "results_group_movie_country", merge_condition, "created_date")

In [0]:
%sql
SELECT * FROM movie_gold.results_group_movie_country

In [0]:
# display(spark.read.parquet(f"{gold_folder_path}/results_group_movie_country"))